In [ ]:
"""
LLM Debate Simulator
---------------------
Simulates a debate between 3 LLM "debaters" powered by local Ollama models.
By few modifications, this script can be adapted to any number of debaters, turns, or paid models.

- Topic is entered at runtime or hardcoded in the script.
- Each debater gets exactly 4 turns (12 turns total).
- Turn order is randomized, with the constraint that no debater goes twice
  in a row (back-to-back), while still respecting the 4-turn budget per debater.
- Each turn, a debater receives the FULL transcript so far and must produce
  a rebuttal to prior discussion + their own argument, self-limited to 250 words
  (instructed in the prompt; not programmatically truncated).
- Output is printed to console and saved to debate_transcript.txt.

Requirements:
    pip install requests
    Ollama must be running locally (default: http://localhost:11434)
"""

import random
from openai import OpenAI

# ---------------------------------------------------------------------------
# CONFIGURATION — set these to the local Ollama model names you have pulled.
# All three can be the same model, or different ones.
# Make changes here to experiment with different models, turn counts, or word limits.
# ---------------------------------------------------------------------------
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

DEBATERS = [
    {"name": "Debater Aria", "model": "gemma2:2b"},
    {"name": "Debater Ben", "model": "qwen3:4b"},
    {"name": "Debater Cyrus", "model": "llama3.2:latest"},
]

TURNS_PER_DEBATER = 4
WORD_LIMIT = 250
TRANSCRIPT_FILE = "debate_transcript.md"


# ---------------------------------------------------------------------------
# Turn order generation
# ---------------------------------------------------------------------------
def generate_turn_order(debater_names, turns_per_debater):
    """
    Generate a random turn order such that:
    - Each debater appears exactly `turns_per_debater` times.
    - No debater appears twice in a row (back-to-back).
    """
    total_turns = len(debater_names) * turns_per_debater
    remaining = {name: turns_per_debater for name in debater_names}
    order = []
    last = None

    for _ in range(total_turns):
        # Candidates: debaters with turns remaining, excluding the last speaker
        # (unless excluding them makes it impossible to continue).
        candidates = [d for d in remaining if remaining[d] > 0 and d != last]

        if not candidates:
            # Fallback safety net (shouldn't normally trigger with 3 debaters,
            # 4 turns each, but guards against edge cases).
            candidates = [d for d in remaining if remaining[d] > 0]

        choice = random.choice(candidates)
        order.append(choice)
        remaining[choice] -= 1
        last = choice

    return order


# ---------------------------------------------------------------------------
# Prompt construction
# ---------------------------------------------------------------------------
def build_prompt(topic, debater_name, transcript_so_far):
    if transcript_so_far.strip() == "":
        history_section = "No arguments have been made yet. You are speaking first."
    else:
        history_section = (
            "Here is the full debate transcript so far:\n\n"
            f"{transcript_so_far}"
        )

    prompt = f"""You are {debater_name}, a participant in a live debate on the topic:

"{topic}"

You may freely choose and argue any position on this topic (for, against, or a nuanced
stance), based on what strengthens your case given the discussion so far.

{history_section}

Instructions for your turn:
1. If prior arguments exist, provide a concise rebuttal addressing points made so far
   by the other debaters.
2. Then present your own argument or supporting points on the topic.
3. Do NOT invent, fabricate, or make up facts, statistics, studies, or sources. Only use
   reasoning, widely known general knowledge, or points already raised in the debate.
   If you are unsure whether something is factually accurate, do not state it as fact.
4. If you genuinely do not have a strong argument or rebuttal to contribute this turn,
   you may skip by responding with EXACTLY this sentence and nothing else:
   "I don't have any arguments and pass my turn"
5. Your ENTIRE response must be a maximum of {WORD_LIMIT} words. Stay within this limit.
6. Do not include stage directions, meta-commentary, or restate these instructions.
7. Speak in your own voice as {debater_name}.
8. Present arguments as bullets, and avoid long sentences or paragraphs. Use clear, concise language.

Now provide your turn:"""
    return prompt


# ---------------------------------------------------------------------------
# Ollama API call
# ---------------------------------------------------------------------------
def call_ollama(model, prompt):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"[ERROR: Could not get response from model '{model}': {e}]"


# ---------------------------------------------------------------------------
# Evaluation phase: debaters rate each other's arguments
# ---------------------------------------------------------------------------
def build_evaluation_prompt(topic, judge_name, other_names, full_transcript):
    others_list = ", ".join(other_names)
    prompt = f"""The debate on the topic "{topic}" has now concluded. Here is the full transcript:

{full_transcript}

You are {judge_name}. You will now act as an impartial judge and evaluate the OTHER
debaters (not yourself): {others_list}.

For each of them, rate the strength of their arguments throughout the debate on a scale
of 1 (worst) to 10 (best), based on logical soundness, relevance, use of rebuttals, and
clarity. Do not favor a debater simply because you agree with their position.

Respond ONLY in the following exact format, with one line per debater you are rating,
and nothing else (no explanations, no extra text):

DebaterName: score

For example:
Debater X: 7
Debater Y: 4

Now provide your ratings for: {others_list}"""
    return prompt


def parse_scores(response_text, expected_names):
    """
    Parse lines like 'Debater Ben: 7' out of the model's response.
    Returns a dict {name: score_int_or_None}.
    """
    scores = {name: None for name in expected_names}
    for line in response_text.splitlines():
        line = line.strip()
        if ":" not in line:
            continue
        name_part, score_part = line.rsplit(":", 1)
        name_part = name_part.strip()
        score_part = score_part.strip()

        for expected_name in expected_names:
            if expected_name.lower() in name_part.lower():
                digits = "".join(ch for ch in score_part if ch.isdigit())
                if digits:
                    try:
                        scores[expected_name] = int(digits)
                    except ValueError:
                        pass
                break
    return scores


def run_evaluation_phase(topic, transcript_so_far, md_lines):
    debater_names = [d["name"] for d in DEBATERS]
    name_to_model = {d["name"]: d["model"] for d in DEBATERS}

    print("\n" + "=" * 60)
    print("EVALUATION PHASE: debaters rate each other (1-10)")
    print("=" * 60 + "\n")

    # results[judge][ratee] = score
    results = {judge: {} for judge in debater_names}

    for judge_name in debater_names:
        other_names = [n for n in debater_names if n != judge_name]
        model = name_to_model[judge_name]
        prompt = build_evaluation_prompt(topic, judge_name, other_names, transcript_so_far)

        print(f"--- {judge_name} ({model}) is evaluating the other debaters... ---")
        response_text = call_ollama(model, prompt)
        print(f"{judge_name}'s raw evaluation:\n{response_text}\n")

        scores = parse_scores(response_text, other_names)
        results[judge_name] = scores

    # --- Build console + markdown table ---
    # Rows = debater being rated, Columns = judge giving the rating
    header_row = ["Debater \\ Judged by"] + debater_names + ["Average"]
    table_rows = []

    for ratee in debater_names:
        row = [ratee]
        collected_scores = []
        for judge in debater_names:
            if judge == ratee:
                row.append("—")  # debaters don't rate themselves
            else:
                score = results[judge].get(ratee)
                if score is not None:
                    row.append(str(score))
                    collected_scores.append(score)
                else:
                    row.append("N/A")
        avg = (sum(collected_scores) / len(collected_scores)) if collected_scores else None
        row.append(f"{avg:.2f}" if avg is not None else "N/A")
        table_rows.append(row)

    # Console print (simple aligned text table)
    print("\nFINAL EVALUATION RESULTS")
    print(" | ".join(header_row))
    for row in table_rows:
        print(" | ".join(row))

    # Markdown table
    md_lines.append("## Peer Evaluation Results")
    md_lines.append("")
    md_lines.append(
        "Each debater rated the *other* debaters' arguments from 1 (worst) to 10 (best)."
    )
    md_lines.append("")
    md_lines.append("| " + " | ".join(header_row) + " |")
    md_lines.append("|" + "|".join(["---"] * len(header_row)) + "|")
    for row in table_rows:
        md_lines.append("| " + " | ".join(row) + " |")
    md_lines.append("")

    return md_lines


# ---------------------------------------------------------------------------
# Main debate loop
# ---------------------------------------------------------------------------
def run_debate(topic):
    name_to_model = {d["name"]: d["model"] for d in DEBATERS}
    debater_names = [d["name"] for d in DEBATERS]

    turn_order = generate_turn_order(debater_names, TURNS_PER_DEBATER)

    transcript_so_far = ""   # full raw transcript, fed back into prompts
    md_lines = []             # markdown-formatted output for the file

    console_header = f"DEBATE TOPIC: {topic}\n" + ("=" * 60) + "\n"
    print(console_header)

    # --- Markdown header ---
    md_lines.append(f"# Debate Transcript")
    md_lines.append("")
    md_lines.append(f"**Topic:** {topic}")
    md_lines.append("")
    md_lines.append("**Participants:**")
    for d in DEBATERS:
        md_lines.append(f"- **{d['name']}** — model: `{d['model']}`")
    md_lines.append("")
    md_lines.append("---")
    md_lines.append("")

    for turn_num, debater_name in enumerate(turn_order, start=1):
        model = name_to_model[debater_name]
        prompt = build_prompt(topic, debater_name, transcript_so_far)

        print(f"\n--- Turn {turn_num}: {debater_name} ({model}) is thinking... ---\n")

        response_text = call_ollama(model, prompt)

        console_block = f"[Turn {turn_num}] {debater_name}:\n{response_text}\n"
        print(console_block)

        # --- Markdown block for this turn ---
        md_lines.append(f"## Turn {turn_num} — {debater_name}")
        md_lines.append("")
        md_lines.append(f"*Model: `{model}`*")
        md_lines.append("")
        # Preserve paragraph breaks from the model's response
        for paragraph in response_text.split("\n"):
            paragraph = paragraph.strip()
            if paragraph:
                md_lines.append(paragraph)
                md_lines.append("")
        md_lines.append("---")
        md_lines.append("")

        transcript_so_far += f"\n{debater_name}: {response_text}\n"

    # --- Footer ---
    md_lines.append("## End of Debate")
    md_lines.append("")
    md_lines.append(f"Total turns: {len(turn_order)} "
                     f"({TURNS_PER_DEBATER} per debater, {len(DEBATERS)} debaters).")
    md_lines.append("")
    md_lines.append("---")
    md_lines.append("")

    # --- Evaluation phase: debaters rate each other ---
    md_lines = run_evaluation_phase(topic, transcript_so_far, md_lines)

    # Write full markdown transcript to file
    with open(TRANSCRIPT_FILE, "w", encoding="utf-8") as f:
        f.write("\n".join(md_lines))

    print(f"\nDebate complete. Full transcript saved to '{TRANSCRIPT_FILE}'.")


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    #topic_input = input("Enter the debate topic: ").strip()
    topic_input = "Which country has had the best performance in all FIFA tournaments?"
    if not topic_input:
        print("No topic entered. Exiting.")
    else:
        run_debate(topic_input)